In [ ]:
import kagglehub,os,pandas as pd
path = kagglehub.dataset_download("rohiteng/amazon-sales-dataset")
print(os.listdir(path))
df=pd.read_csv(path + "/Amazon.csv")

In [ ]:
df = df[df['OrderStatus'].isin(['Delivered', 'Shipped'])]
print(f'Filtered DataFrame shape for Delivered & Shipped: {df.shape}')

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
df.duplicated().sum()

In [ ]:
df['PaymentMethod'].unique()

In [ ]:
import duckdb

In [ ]:
import duckdb

query_city_only = """
SELECT
    City
FROM df
GROUP BY City
ORDER BY SUM(ShippingCost) / SUM(TotalAmount) DESC;
"""

cities_by_shipping_ratio = duckdb.sql(query_city_only).df()
display(cities_by_shipping_ratio)

In [ ]:
import duckdb

query = """
SELECT
    City,
    SUM(ShippingCost) AS Total_Shipping_Cost,
    SUM(TotalAmount) AS Total_Revenue,
    ROUND(SUM(ShippingCost) / SUM(TotalAmount) * 100, 2) AS Shipping_Cost_Percentage
FROM df
GROUP BY City
ORDER BY Total_Shipping_Cost DESC;
"""

city_comparison = duckdb.sql(query).df()
display(city_comparison)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

x_indices = np.arange(len(city_comparison['City']))
ax1.bar(x_indices, city_comparison['Total_Revenue'], color='#1f77b4', alpha=0.75)
ax1.set_title('Total Revenue ($) per City (Flat Distribution)')
ax1.set_ylabel('Revenue ($ in Millions)')
ax1.set_xticks(x_indices)
ax1.set_xticklabels(city_comparison['City'], rotation=45, ha='right')
ax1.set_ylim(0, city_comparison['Total_Revenue'].max() * 1.15)
ax1.get_yaxis().set_major_formatter(FuncFormatter(lambda x, p: f'${x*1e-6:.2f}M'))
ax1.grid(True, linestyle='--', alpha=0.3)

ax2.bar(x_indices, city_comparison['Total_Shipping_Cost'], color='#d62728', alpha=0.75)
ax2.set_title('Total Shipping Cost ($) per City (Flat Distribution)')
ax2.set_ylabel('Shipping Cost ($)')
ax2.set_xticks(x_indices)
ax2.set_xticklabels(city_comparison['City'], rotation=45, ha='right')
ax2.set_ylim(0, city_comparison['Total_Shipping_Cost'].max() * 1.15)
ax2.grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Revenue & Shipping Cost per City (Exposing Artificial Flatness)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

def check_randomness(df, group_col, value_col, iterations=500):
    grouped = df.groupby(group_col)[value_col].sum()
    cv_actual = grouped.std() / grouped.mean()

    cv_random = []
    for i in range(iterations):
        shuffled = df[group_col].sample(frac=1, random_state=i).values
        temp_df = df.assign(**{f'{group_col}_shuffled': shuffled})
        grouped_rand = temp_df.groupby(f'{group_col}_shuffled')[value_col].sum()
        cv_random.append(grouped_rand.std() / grouped_rand.mean())

    p5, p50, p95 = np.percentile(cv_random, [5, 50, 95])

    print(f"Actual CV: {cv_actual:.4f}")
    print(f"Random CV (5%, 50%, 95%): [{p5:.4f}, {p50:.4f}, {p95:.4f}]")

    if cv_actual > p95:
        print("=> Result: This column HAS meaningful information (variance is significantly higher than random).")
    elif cv_actual < p5:
        print("=> Result: This column is FLATTER than random (strong indicator of synthetic/fake data).")
    else:
        print("=> Result: This column is STATISTICALLY RANDOM (no meaningful information).")

In [ ]:
check_randomness(df, 'City', 'TotalAmount')